In [1]:
import pandas as pd
import re
import numpy as np
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm.notebook import tqdm
import time
from google.colab import files
import json
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

In [2]:
pd.set_option('display.max_colwidth', None)
tqdm.pandas()

In [ ]:
df_1 = pd.read_csv("/content/vacancies_1.csv", encoding='utf-8')

df_2 = pd.read_csv("/content/vacancies_2.csv", encoding='utf-8')

df_3 = pd.read_csv("/content/vacancies_3.csv", encoding='utf-8')

In [ ]:
df_all = pd.concat([df_1, df_2, df_3], ignore_index=True)
df = df_all.drop_duplicates(subset=['ID'], keep='first')

print(f"Было строк: {len(df_all)}")
print(f"Стало строк: {len(df)}")

Было строк: 1671
Стало строк: 1201


## Этап 1: отбор вакансий, относящихся к IT сфере

In [ ]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
it_templates = [
    "разработка программного обеспечения на python java go c++",
    "программирование и кодинг frontend react angular vue",
    "тестирование и качество ПО qa manual auto pytest selenium",
    "системное администрирование linux windows server",
    "devops и инфраструктура kubernetes docker ci cd",
    "анализ данных и машинное обучение data science pandas sql",
    "информационная безопасность кибербезопасность soc siem",
    "работа с базами данных sql postgresql clickhouse oracle",
    "web разработка фронтенд бэкенд api rest",
    "мобильная разработка android ios swift kotlin flutter",
    "1С программирование и внедрение разработка конфигураций",
    "работа с серверным и сетевым оборудованием цод",
    "нагрузочное тестирование jmeter gatling",
    "big data hadoop spark kafka airflow",
    "сборка и ремонт серверного оборудования дата-центр",
    "администрирование linux windows active directory",
    "стажер-разработчик go golang микросервисы",
    "стажер data engineer python sql hadoop",
    "junior аналитик данных sql python pandas",
    "стажер по нагрузочному тестированию jmeter java",
    "стажер по продвижению ит-решений технологии программирование",
    "выездной системный администратор серверы цод",
    "программист-стажер 1с разработка конфигураций",
    "стажер бизнес-аналитик ит требования автоматизация"
]

non_it_templates = [
    "продажи и работа с клиентами b2b холодные звонки",
    "маркетинг и реклама smm pr продвижение",
    "бухгалтерский учет и финансы налоги отчетность",
    "кадровое делопроизводство hr рекрутинг",
    "закупки и логистика снабжение тендеры",
    "административная работа офис-менеджер ассистент",
    "юридическое сопровождение договоры суды консультации",
    "управление персоналом hr bp обучение развитие",
    "водитель курьер доставка",
    "работа на складе кладовщик комплектовщик",
    "стажер в юридический департамент договоры суды",
    "стажер менеджера по привлечению клиентов продажи b2b",
    "стажер-ассистент клиентских менеджеров crm документооборот",
    "младший финансовый аналитик бюджетирование отчетность",
    "младший инженер-технолог производство оборудование",
    "стажер по сопровождению юридических лиц архив документы",
    "мобильный банкир доставка клиентам"
]


it_embeddings = model.encode(it_templates, convert_to_tensor=True, show_progress_bar=True)
non_it_embeddings = model.encode(non_it_templates, convert_to_tensor=True, show_progress_bar=True)

STRONG_IT_KEYWORDS = [
    'python', 'java', 'javascript', 'typescript', 'sql', 'golang', 'rust', 'kotlin', 'swift',
    'docker', 'kubernetes', 'git', 'api', 'rest', 'graphql', 'grpc',
    'react', 'angular', 'vue', 'node.js', 'django', 'flask', 'spring', 'hibernate',
    'postgresql', 'mongodb', 'redis', 'clickhouse', 'oracle', 'kafka', 'spark', 'hadoop',
    'jenkins', 'gitlab', 'terraform', 'ansible', 'prometheus', 'grafana',
    'tensorflow', 'pytorch', 'pandas', 'numpy', 'scikit-learn',
    'linux', 'windows server', 'bash', 'powershell', 'nginx', 'apache',
    'ios', 'android', 'flutter', '1с', 'bitrix', 'c#', 'c++', 'go', 'unity',
    'selenium', 'pytest', 'junit', 'postman', 'swagger', 'allure',
    'devops', 'ci/cd', 'data engineer', 'data scientist', 'ml engineer', 'qa engineer',
    'разработчик', 'программист', 'тестировщик', 'аналитик данных', 'системный администратор',
    'администрирование серверов', 'сетевое оборудование', 'цод', 'дата-центр'
]

IT_COMPANIES = [
    'яндекс', 'ozon', 'vk', 'т-банк', 'сбер', 'втб', 'psb', 'псб',
    'лаборатория касперского', 'kaspersky', 'авито', 'positive technologies',
    'крок', 'ibs', 'softline', 'selectel', 't2', 'ростелеком', 'wildberries',
    'lamoda', 'купер', 'самокат', 't1'
]

NON_IT_PATTERNS = [
    'продавец', 'кассир', 'расклейщик', 'водитель такси', 'курьер пеший',
    'уборщик', 'грузчик', 'охранник', 'кладовщик', 'комплектовщик',
    'повар', 'официант', 'бармен', 'горничная', 'промоутер',
    'бухгалтер', 'кадровый специалист', 'юрист', 'офис-менеджер'
]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
def safe_str(value):
    return str(value) if pd.notna(value) else ''

def has_strong_it_keywords(text):
    if not text:
        return False
    text_lower = text.lower()
    for kw in STRONG_IT_KEYWORDS:
        if kw in text_lower:
            return True
    return False

def has_non_it_patterns(text):
    if not text:
        return False
    text_lower = text.lower()
    for pattern in NON_IT_PATTERNS:
        if pattern in text_lower:
            return True
    return False

def is_it_company(text):
    if not text:
        return False
    text_lower = text.lower()
    for company in IT_COMPANIES:
        if company in text_lower:
            return True
    return False

def classify_by_embeddings(text):
    if len(text) < 100:
        return None
    text_emb = model.encode(text, convert_to_tensor=True)
    it_scores = util.cos_sim(text_emb, it_embeddings)[0]
    non_it_scores = util.cos_sim(text_emb, non_it_embeddings)[0]

    top_k = 3
    top_it = torch.topk(it_scores, min(top_k, len(it_scores)))[0]
    top_non_it = torch.topk(non_it_scores, min(top_k, len(non_it_scores)))[0]

    avg_it = torch.mean(top_it).item()
    avg_non_it = torch.mean(top_non_it).item()

    threshold = 0.35

    if avg_it > threshold and avg_it > avg_non_it:
        return 1
    elif avg_non_it > threshold and avg_non_it > avg_it:
        return 0

def classify_vacancy(row):
    title = safe_str(row.get('Название', ''))
    text = safe_str(row.get('Текст', ''))
    full_text = f"{title} {text}".lower()

    if has_non_it_patterns(full_text):
        if not has_strong_it_keywords(full_text):
            return 0

    if has_strong_it_keywords(full_text):
        return 1

    if is_it_company(full_text):
        tech_words = ['разработк', 'тестировани', 'администрировани', 'аналитик',
                      'инженер', 'программист', 'систем']
        for word in tech_words:
            if word in full_text:
                return 1

    if len(text) > 150:
        emb_result = classify_by_embeddings(text)
        if emb_result is not None:
            return emb_result

    return 0

In [ ]:
tqdm.pandas(desc="Классификация")
df['is_it'] = df.progress_apply(classify_vacancy, axis=1)

df.groupby('is_it').count()

Классификация: 100%|██████████| 1201/1201 [00:05<00:00, 209.57it/s]
/tmp/ipykernel_28392/1405525857.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['is_it'] = df.progress_apply(classify_vacancy, axis=1)


,Unnamed: 0,ID,Название,Департамент,Город,Опыт,Расписание,Формат работы,Рабочие часы,График,Зарплата,Дата публикации,Ссылка,Текст,ID компании
is_it,,,,,,,,,,,,,,,
0,443,443,443,294,443,443,443,392,443,443,443,443,443,431,443
1,758,758,758,436,758,758,758,688,758,750,758,758,758,756,758


In [ ]:
df_sample = df.sample(n=100, random_state=42)

In [ ]:
df_sample.to_excel('data_test.xlsx', index=False)

In [ ]:
df_sample_target = pd.read_excel('/content/data_test_target.xlsx')

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

y_true = df_sample_target['is_it']
y_pred = df_sample['is_it']
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("="*50)
print("МЕТРИКИ КАЧЕСТВА МОДЕЛИ")
print("="*50)
print(f"Accuracy (общая точность):  {accuracy:.3f} ({accuracy*100:.1f}%)")
print(f"Precision (точность IT):    {precision:.3f} ({precision*100:.1f}%)")
print(f"Recall (полнота IT):        {recall:.3f} ({recall*100:.1f}%)")
print(f"F1-score:                   {f1:.3f}")
print("="*50)

print("\nМатрица ошибок:")
print("            Предсказано")
print("            Non-IT   IT")
cm = confusion_matrix(y_true, y_pred)
print(f"Реально     Non-IT   {cm[0,0]:5d}  {cm[0,1]:5d}")
print(f"            IT       {cm[1,0]:5d}  {cm[1,1]:5d}")

print("\n" + "="*50)
print("ДЕТАЛЬНЫЙ ОТЧЕТ ПО КЛАССАМ:")
print(classification_report(y_true, y_pred, target_names=['Non-IT', 'IT']))

МЕТРИКИ КАЧЕСТВА МОДЕЛИ
Accuracy (общая точность):  0.940 (94.0%)
Precision (точность IT):    0.938 (93.8%)
Recall (полнота IT):        0.968 (96.8%)
F1-score:                   0.953

Матрица ошибок:
            Предсказано
            Non-IT   IT
Реально     Non-IT      33      4
            IT           2     61

ДЕТАЛЬНЫЙ ОТЧЕТ ПО КЛАССАМ:
              precision    recall  f1-score   support

      Non-IT       0.94      0.89      0.92        37
          IT       0.94      0.97      0.95        63

    accuracy                           0.94       100
   macro avg       0.94      0.93      0.93       100
weighted avg       0.94      0.94      0.94       100



**Общая оценка качества модели**

Модель показала результат в 94% точности. Это означает, что из каждых 100 вакансий модель правильно классифицирует 94, ошибаясь лишь в 6 случаях. Такой показатель говорит о том, что модель хорошо справляется со своей задачей.

**Как модель работает с IT-вакансиями**

Точность составляет 93,8%: когда модель определяет вакансию как IT, она права в 94 случаях из 100.
Из 63 реальных IT-вакансий модель распознала 61 и пропустила только 2.

**Как модель работает с non-IT-вакансиями**

Из 37 реальных non-IT-вакансий модель правильно определила 33.Только 4 non-IT-вакансии ошибочно попали в IT, и только 2 IT-вакансии ошибочно отнесены к non-IT. В итоговой выборке с признаком is_it = 1 будут перепроверятся все вакансии вручную, все вакансии non-it будут удалены

## Этап 2: разметка признаков

In [ ]:
df_analysis = df[df['is_it'] == 1]

In [ ]:
df_analysis.to_excel('it_vacancies.xlsx')

In [3]:
df_analysis_it = pd.read_excel('it_vacancies_result.xlsx')

In [4]:
df_coded = pd.DataFrame()

df_coded['A0'] = df_analysis_it['ID']
df_coded['A2'] = df_analysis_it['Название']
df_coded['A3'] = df_analysis_it['Департамент']
df_coded['A4'] = df_analysis_it['Город']
df_coded['A5'] = df_analysis_it['Опыт']
df_coded['A6'] = df_analysis_it['Расписание']
df_coded['A7'] = df_analysis_it['Формат работы']
df_coded['A8'] = df_analysis_it['Департамент']
df_coded['A9'] = df_analysis_it['График']
df_coded['A10'] = df_analysis_it['Зарплата']
df_coded['A11'] = df_analysis_it['Дата публикации']
df_coded['A12'] = df_analysis_it['Ссылка']

In [5]:
def get_level(row):
    title = str(row['Название']).lower()
    if 'стажёр' in title or 'стажер' in title or 'trainee' in title or 'intern' in title:
        return 'Стажер'
    elif 'junior' in title or 'младший' in title or 'начинающий' in title:
        return 'Junior'
    else:
        return 'Не указан'

df_coded['A1'] = df_analysis_it.apply(get_level, axis=1)

In [6]:
df_coded.head(1)

,A0,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A1
0,129605023,Стажёр в IT-команду дата-центра (ЦОД) Яндекса,Yandex Infrastructure,Владимир,Нет опыта,Полный день,На месте работодателя,Yandex Infrastructure,5/2,Не указана,2026-01-19T15:22:30+0300,https://hh.ru/vacancy/129605023,Стажер


In [7]:
def process_b1(row):
    schedule = str(row.get('Расписание', '')).lower() if pd.notna(row.get('Расписание')) else ''

    if schedule and re.search(r'гибкий|свободный|плавающий|удобный', schedule):
        return 1
    text = str(row.get('Текст', '')).lower() if pd.notna(row.get('Текст')) else ''
    if re.search(r'гибкий\s*график|свободный\s*график|плавающий\s*график|'
                 r'самостоятельно\s*планируешь|нет\s*фиксированного|'
                 r'удобный\s*график|график\s*под\s*тебя|'
                 r'свободное\s*начало|утреннее\s*окно', text):
        return 1

    return 0

def process_b2(row):


    text = str(row.get('Текст', '')).lower() if pd.notna(row.get('Текст')) else ''
    if re.search(r'полностью\s*удален|100%\s*удален|только\s*удален|'
                 r'работа\s*из\s*дома|work\s*from\s*home|полная\s*удаленка', text):
        return 1
    if re.search(r'гибрид|mix|\d+\s*дня\s*в\s*офисе|смешанный\s*формат|'
                 r'часть\s*дней\s*в\s*офисе|несколько\s*дней\s*в\s*офисе', text):
        return 2
    if re.search(r'возможна\s*удаленка|частичн\S*\s*удален|'
                 r'гибкий\s*формат|обсуждаем\s*формат|'
                 r'формат\s*работы\s*обсужда|удаленка\s*обсужда', text):
        return 3
    if re.search(r'только\s*офис|работа\s*в\s*офисе|офис\s*в\s*москве|'
                 r'присутствие\s*в\s*офисе|5/2\s*в\s*офисе', text):
        return 0
    return 0


def process_c1(row):
    text = str(row.get('Текст', '')).lower() if pd.notna(row.get('Текст')) else ''

    if re.search(r'дмс|добровольн\S*\s*медицинск\S*\s*страхован|'
                 r'медицинск\S*\s*страховк\S*|полис\s*дмс|'
                 r'расширенн\S*\s*дмс|дмс\s*со\s*стоматологией|'
                 r'дмс\s*с\s*первого\s*дня|мед\s*страховка', text):
        return 1
    return 0

def process_c2(row):
    text = str(row.get('Текст', '')).lower() if pd.notna(row.get('Текст')) else ''

    if re.search(r'фитнес|спортзал|тренажерн\S*\s*зал|корпоративн\S*\s*спорт|'
                 r'компенсац\S*\s*фитнес|спортивн\S*\s*секц\S*|'
                 r'all\s*sports|спорт\s*за\s*счет|абонемент\s*в\s*спортзал|'
                 r'йога|бассейн|спортивн\S*\s*мероприят\S*', text):
        return 1
    return 0

def process_c3(row):
    text = str(row.get('Текст', '')).lower() if pd.notna(row.get('Текст')) else ''

    if re.search(r'оплата\s*питани[яе]|доставк[ау]\s*обедов|бизнес-ланч|'
                 r'бесплатн\S*\s*обед[ыы]|кофе|чай|снэки|'
                 r'собственн\S*\s*кафе|компенсац\S*\s*питани|'
                 r'завтрак[и]|обед[ы]\s*за\s*счет|корпоративн\S*\s*столов|'
                 r'вкусн\S*\s*кормят|сладкост[и]|фрукт[ы]', text):
        return 1
    return 0

def process_c4(row):
    text = str(row.get('Текст', '')).lower() if pd.notna(row.get('Текст')) else ''

    if re.search(r'предоставим\s*ноутбук|рабочий\s*ноутбук|выдаем\s*технику|'
                 r'mac|macbook|монитор|два\s*монитора|современн\S*\s*техника|'
                 r'компенсац\S*\s*на\s*покупк\S*\s*техник|компенсируем\s*технику|'
                 r'оргтехника|рабочее\s*оборудование|техника\s*предоставля|'
                 r'мощн\S*\s*компьютер|рабоч\S*\s*станц\S*', text):
        return 1
    return 0

tech_keywords = [
    'python', 'java', 'kotlin', 'swift', 'go', 'golang', 'rust', 'c#', 'c++',
    'javascript', 'typescript', 'php', 'ruby', 'scala', 'r', 'dart',
    'react', 'vue', 'angular', 'node\\.js', 'django', 'flask', 'fastapi', 'spring',
    'tensorflow', 'pytorch', 'keras', 'pandas', 'numpy',
    'sql', 'postgresql', 'mysql', 'mongodb', 'redis', 'clickhouse',
    'docker', 'kubernetes', 'k8s', 'git', 'ci/cd', 'jenkins', 'ansible',
    'aws', 'gcp', 'azure', 'cloud',
    'llm', 'ml', 'ai', 'машинное обучение', 'искусственный интеллект',
    'nlp', 'computer vision', 'data science', 'spark', 'kafka'
]

def process_f3(row):
    text = str(row.get('Текст', '')).lower() if pd.notna(row.get('Текст')) else ''
    pattern = r'\b(?:' + '|'.join(re.escape(kw) for kw in tech_keywords) + r')\b'

    if re.search(pattern, text):
        return 1
    return 0


df_coded['B1'] = df_analysis_it.apply(process_b1, axis=1)
df_coded['B2'] = df_analysis_it.apply(process_b2, axis=1)
df_coded['C1'] = df_analysis_it.apply(process_c1, axis=1)
df_coded['C2'] = df_analysis_it.apply(process_c2, axis=1)
df_coded['C3'] = df_analysis_it.apply(process_c3, axis=1)
df_coded['C4'] = df_analysis_it.apply(process_c4, axis=1)
df_coded['F3'] = df_analysis_it.apply(process_f3, axis=1)


In [ ]:
from transformers import BitsAndBytesConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_name = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1000,
    pad_token_id=tokenizer.eos_token_id,
    temperature=0.1
)

In [9]:
model.generation_config.do_sample = False
model.generation_config.max_new_tokens = 250

In [14]:
df_coded['Текст'] = df_analysis_it['Текст']

In [13]:
llm_columns = ['B3', 'B4', 'D1', 'D2', 'D3', 'D4', 'E1', 'E2', 'E3', 'E4', 'F1', 'F2']

for col in llm_columns:
    df_coded[col] = 0

In [17]:
SYSTEM_PROMPT = """Ты — эксперт по анализу IT-вакансий. Твоя задача — максимально точно извлечь признаки из текста вакансии.

ПРАВИЛА:
1. ВАЖНО: Если признак явно не указан в тексте, ставь значение по умолчанию (0 или минимальное значение).
2. НЕ ДОМЫСЛИВАЙ информацию, которой нет в тексте, но приведенные далее фразы для признаков могут быть сформулированы немного иначе (их смысл должен быть аналогичным).
3. Возвращай ТОЛЬКО JSON объект, без пояснений, без markdown-разметки, без дополнительного текста.
4. Все значения должны быть числами (int), не строками.

ПРИЗНАКИ:

Блок B (Режим работы):
- B3: оплата переработок (1 = есть: "оплата сверхурочных", "компенсация переработок", "переработки оплачиваются")
- B4: дополнительный отпуск (1 = есть: "дополнительный отпуск", "отгулы", "дополнительные выходные", "3 дополнительных дня к отпуску", "extra days off")

Блок D (Развитие):
- D1: обучение за счёт компании (1 = есть: "оплата курсов", "тренинги", "обучение", "корпоративный университет", "бесплатное обучение", "онлайн и офлайн обучение", "курсы и сертификации", "доступ к библиотеке", "международные сертификации", "английский язык", "оплатим обучение")
- D2: наставник/ментор (1 = есть: "ментор", "наставник", "buddy", "наставничество", "персональный наставник", "поддержка ментора", "опытные коллеги")
- D3: карьерный рост (1 = есть: "карьерный рост", "performance review", "вертикальный и горизонтальный рост", "повышение", "переход в штат", "ротация в другой продукт", "матрица компетенций", "регулярное ревью")
- D4: конференции (1 = есть: "участие в конференциях", "посещение конференций", "митапы", "профессиональные мероприятия", "Jet Security Conference", "выступление на конференциях", "возможность быть спикером", "публичные выступления", "пишем статьи на Хабр")

Блок E (Коллектив и условия):
- E1: команда/сообщество (1 = есть: "дружная команда", "молодой коллектив", "команда профессионалов", "сильное комьюнити", "эксперты в своей области", "обмен опытом", "поддержка коллег")
- E2: мероприятия (1 = есть: "тимбилдинг", "корпоративы", "корпоративные мероприятия", "нетворкинг", "неформальные мероприятия", "творческие вечера", "киберлига", "волонтерство", "клубы по интересам")
- E3: офисная инфраструктура (1 = есть: "современный офис", "релакс-зона", "кофейни", "кухня", "лаунж-зоны", "комнаты для сна", "терраса", "спортзал в офисе", "массажные кресла", "библиотека", "мед.кабинет", "массаж")
- E4: волонтёрство/КСО (1 = есть: "волонтёрство", "донорство", "благотворительность", "волонтерство", "социальные проекты", "КСО")

Блок F (Карьерные возможности):
- F1: стажировка с трудоустройством (1 = есть: "стажировка с трудоустройством", "оффер по итогам", "переход в штат после стажировки", "возможность дальнейшего трудоустройства", "трудоустройство после стажировки"), если это не стажировка, то следует ставить 0
- F2: быстрый рост (1 = есть: "быстрый рост", "интенсивное обучение", "быстрое развитие", "активный рост", "стажёр может стать руководителем за два года", "быстрый результат")

Примеры входов и выходов:

Пример 1: "Гибкий график. Полная удаленка. ДМС. Оплачиваем фитнес. Есть ментор. Python, Docker, Kubernetes."
Выход: {"B3":0,"B4":0,"D1":0,"D2":1,"D3":0,"D4":0,"E1":0,"E2":0,"E3":0,"E4":0,"F1":0,"F2":0}

Пример 2: "Работа в офисе или гибрид. ДМС со стоматологией. Оплата обедов. Карьерный рост. Участие в конференциях."
Выход: {"B3":0,"B4":0,"D1":0,"D2":0,"D3":1,"D4":1,"E1":0,"E2":0,"E3":0,"E4":0,"F1":0,"F2":0}

Пример 3: "Стажировка с трудоустройством. Наставник. Современный офис с кухней и кофе. Дружная команда. Тимбилдинги."
Выход: {"B3":0,"B4":0,"D1":0,"D2":1,"D3":0,"D4":0,"E1":1,"E2":1,"E3":1,"E4":0,"F1":1,"F2":0}

Пример 4: "Полностью удаленный формат обсуждать не готовы, важно присутствие в офисе несколько дней в неделю. Бесплатное обучение: курсы, митапы. 2 спортзала и сауна прямо в офисе."
Выход: {"B3":0,"B4":0,"D1":1,"D2":0,"D3":0,"D4":1,"E1":0,"E2":0,"E3":1,"E4":0,"F1":0,"F2":0}

ТЕПЕРЬ ОТВЕТЬ ТОЛЬКО JSON ДЛЯ ЭТОЙ ВАКАНСИИ:"""

def classify_with_llm(text, max_retries=2):
    """Классифицирует одну вакансию через LLM"""
    if pd.isna(text) or len(str(text)) < 30:
        return None

    text = str(text)[:3000]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Вакансия:\n{text}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    for attempt in range(max_retries):
        try:
            outputs = pipe(prompt, max_new_tokens=250)
            response = outputs[0]['generated_text'][len(prompt):].strip()
            json_match = re.search(r'\{[^{}]*\}', response, re.DOTALL)
            if json_match:
                result = json.loads(json_match.group())
                for key in result:
                    result[key] = int(result[key])
                return result
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"Ошибка: {e}")
                return None
            time.sleep(1)

    return None

In [ ]:
success_count = 0
fail_count = 0

for idx in tqdm(range(len(df_coded)), desc="LLM классификация"):
    try:
        text = df_coded.iloc[idx]['Текст']
        result = classify_with_llm(text)

        if result:
            for key, value in result.items():
                if key in df_coded.columns:
                    df_coded.loc[df_coded.index[idx], key] = value
            success_count += 1
        else:
            fail_count += 1
    except Exception as e:
        print(f"\nОшибка на индексе {idx}: {e}")
        fail_count += 1

print(f"\nКлассификация завершена")
print(f"   Успешно: {success_count}/{len(df_coded)}")
print(f"   Ошибок: {fail_count}")

In [19]:
df_coded.to_excel('data_base.xlsx')

#### **Валидация автоматического кодирования**


In [23]:
test = pd.read_excel("/content/vacancies_coded_llm_test.xlsx")

In [24]:
target = pd.read_excel("/content/vacancies_coded_llm_target.xlsx")

In [25]:
from sklearn.metrics import cohen_kappa_score, accuracy_score, confusion_matrix

In [28]:
exclude = ['A0', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'Текст']
cols = [c for c in test.columns if c not in exclude]

test_sorted = test.sort_values('A0').reset_index(drop=True)
target_sorted = target.sort_values('A0').reset_index(drop=True)

for col in cols:
    y_true = target_sorted[col]
    y_pred = test_sorted[col]

    acc = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    print(f"{col}: accuracy={acc:.3f}, kappa={kappa:.3f}")

all_true = np.concatenate([target_sorted[col].values for col in cols])
all_pred = np.concatenate([test_sorted[col].values for col in cols])

print(f"\nОбщая accuracy: {accuracy_score(all_true, all_pred):.4f}")
print(f"Общая Cohen's Kappa: {cohen_kappa_score(all_true, all_pred):.4f}")

Unnamed: 0: accuracy=1.000, kappa=1.000
B1: accuracy=0.950, kappa=0.773
B2: accuracy=0.950, kappa=0.890
B3: accuracy=1.000, kappa=nan
B4: accuracy=1.000, kappa=1.000
C1: accuracy=1.000, kappa=1.000
C2: accuracy=0.950, kappa=0.894
C3: accuracy=1.000, kappa=1.000
C4: accuracy=0.900, kappa=0.615
F3: accuracy=1.000, kappa=1.000
D1: accuracy=1.000, kappa=1.000
D2: accuracy=1.000, kappa=1.000
D3: accuracy=1.000, kappa=1.000
D4: accuracy=1.000, kappa=1.000
E1: accuracy=1.000, kappa=1.000
E2: accuracy=1.000, kappa=1.000
E3: accuracy=1.000, kappa=1.000
E4: accuracy=1.000, kappa=nan
F1: accuracy=1.000, kappa=1.000
F2: accuracy=1.000, kappa=1.000

Общая accuracy: 0.9875
Общая Cohen's Kappa: 0.9721


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


#### Результаты валидации кодирования признаков

Проведённая оценка качества работы модели по кодированию признаков (исключая блок А) показала **высокий уровень точности**. Общая accuracy составила **98,75%**, а общий коэффициент согласия Коэна (Cohen's Kappa) — **0,9721**.

### Анализ по отдельным признакам

**Абсолютно точное кодирование** (accuracy = 1.000, kappa = 1.000) продемонстрировано для 16 признаков: `Unnamed: 0`, `B3`, `B4`, `C1`, `C3`, `F3`, `D1`, `D2`, `D3`, `D4`, `E1`, `E2`, `E3`, `F1`, `F2`. Особо стоит отметить признак `E4`, где kappa не определён (nan) из-за отсутствия вариативности, но accuracy = 1.000, что также является идеальным результатом.

**Хорошее качество** показали признаки:
- **B1** (accuracy = 0,950, kappa = 0,773) — высокая, но умеренная согласованность
- **B2** (accuracy = 0,950, kappa = 0,890) — высокая согласованность
- **C2** (accuracy = 0,950, kappa = 0,894) — высокая согласованность
- **C4** (accuracy = 0,900, kappa = 0,615) — наименее надёжный признак

### Интерпретация каппы Коэна

Значения каппы распределились следующим образом:
- **0,89–0,90** (B2, C2) — почти идеальное согласие
- **0,77** (B1) — существенное согласие
- **0,62** (C4) — умеренное согласие

### Выводы

Модель демонстрирует **хорошее качество кодирования** для подавляющего большинства признаков. Единственная проблемная зона - признак **C4**, где точность составляет 90%, а каппа указывает на умеренную согласованность. В целом, модель пригодна для использования.

> Каппа Коэна — это статистическая мера согласованности между двумя оценщиками (или между моделью и ручным кодированием, в данном случае), которая учитывает случайные совпадения.